In [4]:
from langchain_openai import AzureChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_core.prompts import PromptTemplate
from dotenv import load_dotenv
import os

load_dotenv()

AZURE_OPENAI_API_KEY = os.getenv("AZURE_OPENAI_API_KEY")
AZURE_OPENAI_API_VERSION = os.getenv("AZURE_OPENAI_API_VERSION")
AZURE_OPENAI_API_ENDPOINT = os.getenv("AZURE_OPENAI_API_ENDPOINT")
AZURE_OPENAI_API_DEPLOYMENT_4O = os.getenv("AZURE_OPENAI_API_DEPLOYMENT_4O")

llm_openai = AzureChatOpenAI(
    azure_endpoint=AZURE_OPENAI_API_ENDPOINT,
    deployment_name=AZURE_OPENAI_API_DEPLOYMENT_4O,
    openai_api_version=AZURE_OPENAI_API_VERSION,
    openai_api_key=AZURE_OPENAI_API_KEY
)





# Guiding in Prompts

In [5]:
result = llm_openai.invoke("Tell me a joke. Generate the output in key-value pair format with the following keys: setup, punchline.")
result.content

'setup: Why did the scarecrow win an award?  \npunchline: Because he was outstanding in his field!'

# Using Pydantic Models

In [6]:
from pydantic import BaseModel, Field

class llm_schema(BaseModel):
    setup: str = Field(description="The setup for the joke")
    punchline: str = Field(description="The punchline for the joke")


In [8]:
llm_structured_output = llm_openai.with_structured_output(llm_schema)

result = llm_structured_output.invoke("Tell me a joke")
result.punchline

'Because he was outstanding in his field!'

# Practice


In [ ]:
from langchain_openai import AzureChatOpenAI
from pydantic import BaseModel, Field
from dotenv import load_dotenv
import os
import warnings

warnings.filterwarnings("ignore")

# ============================================================
# Initialize environment variables and Azure OpenAI client
# ============================================================

load_dotenv()

AZURE_OPENAI_API_KEY = os.getenv("AZURE_OPENAI_API_KEY")
AZURE_OPENAI_API_VERSION = os.getenv("AZURE_OPENAI_API_VERSION")
AZURE_OPENAI_API_ENDPOINT = os.getenv("AZURE_OPENAI_API_ENDPOINT")
AZURE_OPENAI_API_DEPLOYMENT_4O = os.getenv("AZURE_OPENAI_API_DEPLOYMENT_4O")

llm_openai = AzureChatOpenAI(
    azure_endpoint=AZURE_OPENAI_API_ENDPOINT,
    deployment_name=AZURE_OPENAI_API_DEPLOYMENT_4O,
    openai_api_version=AZURE_OPENAI_API_VERSION,
    openai_api_key=AZURE_OPENAI_API_KEY
)

class Movie(BaseModel):
    title: str = Field(description="The title of the movie")
    genre: str = Field(description="The genre of the movie")
    rating: float = Field(description="Rating from 0 to 10")

movie_llm = llm_openai.with_structured_output(Movie)

# Return a Json with:
# - title: The title of the movie
# - genre: The genre of the movie
# - rating: Rating from 0 to 10 

movie_result = movie_llm.invoke("Recommend me a movie")

print("\n=== Recommended Movie ===")
print(movie_result)
print("Title:", movie_result.title)
print("Rating type:", type(movie_result.rating))
print("Genre:", movie_result.genre)

# Order schema example
class Order(BaseModel):
    product_name: str = Field(description="The name of the product")
    quantity: int = Field(description="The quantity of the product")
    address: str = Field(description="The shipping address for the order")

order_llm = llm_openai.with_structured_output(Order)

order_input = "I want to buy 3 iPhones and ship to Taipei Main Station"

order_result = order_llm.invoke(order_input)

print("\n=== Order Details ===")
print(order_result)
print("Product Name:", order_result.product_name)
print("Quantity:", order_result.quantity)
print("Address:", order_result.address)

# Intention schema example
class Intent(BaseModel):
    intent: str = Field(description="refund | product_question | complaint")

intent_llm = llm_openai.with_structured_output(Intent)

intent_input = "This product is broken, I want my money back"
intent_result = intent_llm.invoke(intent_input)

print("\n=== Detected Intent ===")
print(intent_result)
print("Intent:", intent_result.intent)

# RAG
class Answer(BaseModel):
    answer: str = Field(description="The answer to the question based on the retrieved documents")
    source: str = Field(description="The source of the information used to answer the question")
    confidence: float = Field(description="The confidence score of the answer from 0 to 1")

answer_llm = llm_openai.with_structured_output(Answer)

answer_result = answer_llm.invoke("Explain what is LangChain")

print("\n=== RAG Answer ===")
print(answer_result)
print("Answer:", answer_result.answer)
print("Source:", answer_result.source)
print("Confidence:", answer_result.confidence)

class WeatherArgs(BaseModel):
    localtion: str = Field(description="City name")

# Tool Calling
class ToolCall(BaseModel):
    tool_name: str = Field(description="The name of the tool to call")
    arguments: WeatherArgs

tool_llm = llm_openai.with_structured_output(ToolCall)

tool_input = "What's the weather in Tokyo?"
tool_result = tool_llm.invoke(tool_input)

print("\n=== Tool Call ===")
print(tool_result)
print("Tool Name:", tool_result.tool_name)
print("Arguments:", tool_result.arguments)


=== Recommended Movie ===
title='Knives Out' genre='Mystery/Comedy' rating=8.1
Title: Knives Out
Rating type: <class 'float'>
Genre: Mystery/Comedy

=== Order Details ===
product_name='iPhone' quantity=3 address='Taipei Main Station'
Product Name: iPhone
Quantity: 3
Address: Taipei Main Station

=== Detected Intent ===
intent='refund'
Intent: refund

=== RAG Answer ===
answer='LangChain is an open-source framework designed for developing applications powered by large language models (LLMs). It provides tools and abstractions that make it easy to build complex workflows involving LLMs, such as question answering, summarization, chatbots, and more. LangChain helps developers connect LLMs with various data sources, APIs, and enables features like memory (for context retention) and reasoning chains (multi-step logic). It is widely used for creating advanced AI applications that combine LLMs with external data and services.' source='https://www.langchain.com/how-it-works' confidence=0.95
A

# Pydantic + Validation Error Handling


In [3]:
from langchain_openai import AzureChatOpenAI
from pydantic import BaseModel, Field, ValidationError
from dotenv import load_dotenv
import os
import warnings

warnings.filterwarnings("ignore")

load_dotenv()

AZURE_OPENAI_API_KEY = os.getenv("AZURE_OPENAI_API_KEY")
AZURE_OPENAI_API_VERSION = os.getenv("AZURE_OPENAI_API_VERSION")
AZURE_OPENAI_API_ENDPOINT = os.getenv("AZURE_OPENAI_API_ENDPOINT")
AZURE_OPENAI_API_DEPLOYMENT_4O = os.getenv("AZURE_OPENAI_API_DEPLOYMENT_4O")

llm_openai = AzureChatOpenAI(
    azure_endpoint=AZURE_OPENAI_API_ENDPOINT,
    deployment_name=AZURE_OPENAI_API_DEPLOYMENT_4O,
    openai_api_version=AZURE_OPENAI_API_VERSION,
    openai_api_key=AZURE_OPENAI_API_KEY
)

class Movie(BaseModel):
    title: str = Field(description="The title of the movie")
    genre: str = Field(description="The genre of the movie")
    rating: float = Field(description="Rating")

def call_llm_with_retry(prompt, max_retries=3):
    structured_llm = llm_openai.with_structured_output(Movie)

    for attempt in range(max_retries):
        try:
            print(f"\nAttempt {attempt + 1}")

            result = structured_llm.invoke(prompt)

            if not (0 <= result.rating <= 10):
                raise ValueError(f"Rating out of bounds: {result.rating}")
            
            return result
        
        except(ValidationError, ValueError) as e:
            print(f"Validation error: {e}")

            prompt = f"""
            Fix the output to match this schema strictly:
            - title: string
            - genre: string
            - rating: float between 0 and 10

            Original request: {prompt}
            """

    raise Exception("Failed after retries")

bad_prompt = "Recommend me a movie with rating out of 100"

result = call_llm_with_retry(bad_prompt)

print("\n=== Final Result ===")
print(result)
print(type(result.rating))




Attempt 1
Validation error: Rating out of bounds: 91.0

Attempt 2

=== Final Result ===
title='The Shawshank Redemption' genre='Drama' rating=9.3
<class 'float'>


# Tenaticy

In [ ]:
from langchain_openai import AzureChatOpenAI
from pydantic import BaseModel, Field, ValidationError
from tenacity import retry, stop_after_attempt, wait_exponential, RetryCallState
from dotenv import load_dotenv
import os
import time
import logging

load_dotenv()

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

AZURE_OPENAI_API_KEY = os.getenv("AZURE_OPENAI_API_KEY")
AZURE_OPENAI_API_VERSION = os.getenv("AZURE_OPENAI_API_VERSION")
AZURE_OPENAI_API_ENDPOINT = os.getenv("AZURE_OPENAI_API_ENDPOINT")
AZURE_OPENAI_API_DEPLOYMENT_4O = os.getenv("AZURE_OPENAI_API_DEPLOYMENT_4O")

llm_openai = AzureChatOpenAI(
    azure_endpoint=AZURE_OPENAI_API_ENDPOINT,
    deployment_name=AZURE_OPENAI_API_DEPLOYMENT_4O,
    openai_api_version=AZURE_OPENAI_API_VERSION,
    openai_api_key=AZURE_OPENAI_API_KEY
)

class Movie(BaseModel):
    title: str = Field(description="The title of the movie")
    genre: str = Field(description="The genre of the movie")
    rating: float = Field(description="Rating from 0 to 10")

class Metrics:
    def __init__(self):
        self.total_calls = 0
        self.success = 0
        self.fail = 0
        self.total_retry = 0
        self.total_latency = 0


    def report(self):
        print("\n=== Metrics Report ===")
        print(f"Total Calls: {self.total_calls}")
        print(f"Success: {self.success}")
        print(f"Fail: {self.fail}")
        print(f"Avg retries: {self.total_retry / max(self.total_calls, 1):.2f}")
        print(f"Avg latency: {self.total_latency / max(self.total_calls, 1):.2f} seconds")

metrics = Metrics()

def build_fix_prompt(original_prompt, error_message):
    return f"""
    You must return valid JSON matching this schema:
    - title: string
    - genre: string
    - rating: float between 0 and 10

    Previous prompt: {error_message}

    Original request: {original_prompt}

    Return ONLY valid JSON
    """

def log_retry(retry_state: RetryCallState):
    logger.warning(f"Retry #{retry_state.attempt_number} due to : {retry_state.outcome.exception()}")

    metrics.total_retry += 1

def create_llm_chain():
    return llm_openai.with_structured_output(Movie)

def validate_result(result: Movie):
    if not (0 <= result.rating <= 10):
        raise ValueError(f"Invalid Rating: {result.rating}")
    
def run_llm(prompt: str):
    structured_llm = create_llm_chain()
    return structured_llm.invoke(prompt)

@retry(stop=stop_after_attempt(3), wait=wait_exponential(multiplier=1), after=log_retry, reraise=True)
def robust_llm_call(prompt: str):
    try:
        result = run_llm(prompt)

        validate_result(result)
        
        return result
    
    except (ValidationError, ValueError) as e:
        logger.error(f"ValidationError: {e}")

        fixed_prompt = build_fix_prompt(prompt, str(e))

        logger.info(f"Auto-fixing prompt and retrying...")
        return run_llm(fixed_prompt)
    
def call_with_metrics(prompt: str):
    start = time.time()
    metrics.total_calls += 1

    try:
        result = robust_llm_call(prompt)

        metrics.success += 1
        return result
    
    except Exception as e:
        metrics.fail += 1
        logger.error(f"final failure: {e}")
        return None
    
    finally:
        latency = time.time() - start
        metrics.total_latency += latency
        logger.info(f"Latency: {latency:.2f} seconds")


if __name__ == "__main__":
    bad_prompt = "Recommend a movie with rating out of 100"
    result = call_with_metrics(bad_prompt)

    print("\n=== Final Result ===")
    print(result)
    metrics.report()




INFO:httpx:HTTP Request: POST https://sh-ai-foundry-resource.openai.azure.com/openai/deployments/gpt-4.1/chat/completions?api-version=2024-12-01-preview "HTTP/1.1 200 OK"
ERROR:__main__:ValidationError: Invalid Rating: 95.0
INFO:__main__:Auto-fixing prompt and retrying...
INFO:httpx:HTTP Request: POST https://sh-ai-foundry-resource.openai.azure.com/openai/deployments/gpt-4.1/chat/completions?api-version=2024-12-01-preview "HTTP/1.1 200 OK"
INFO:__main__:Latency: 2.51 seconds



=== Final Result ===
title='Inception' genre='Science Fiction' rating=8.8

=== Metrics Report ===
Total Calls: 1
Success: 1
Fail: 0
Avg retries: 0.00
Avg latency: 2.51 seconds


# Tenacity Practice

In [6]:
# from tenacity import retry

# counter = {"counter": 0}

# @retry
# def unstable_function():
#     counter["counter"] += 1
#     print(f"Call #{counter['counter']}")

#     if counter["counter"] < 3:
#         raise ValueError("Fail!")
    
#     return "Success!"

# print(unstable_function())

# from tenacity import retry, stop_after_attempt

# counter = {"counter": 0}

# @retry(stop=stop_after_attempt(3))
# def always_fails():
#     counter["counter"] += 1
#     print(f"Attempt #{counter['counter']}")
#     raise Exception("Still failing...")

# always_fails()

# from tenacity import retry, stop_after_attempt, wait_exponential
# import time

# @retry(wait=wait_exponential(multiplier=1), stop=stop_after_attempt(1))
# def slow_fail():
#     print("Trying...")
#     raise Exception("Failed!")

# start = time.time()

# try:
#     slow_fail()
# except:
#     pass

# print("Total time:", time.time() - start)

# from tenacity import retry, retry_if_exception_type

# @retry(retry=retry_if_exception_type(ValueError))
# def sometimes_fails(x):
#     print("Input:", x)

#     if x == 1 :
#         raise ValueError("Retry this")
#     elif x ==2:
#         raise TypeError("Don't retry this")
    
#     return "OK"

# print(sometimes_fails(2))


# from tenacity import retry, retry_if_result

# def is_bad(result):
#     return result < 0

# counter = {"counter": 0}

# @retry(retry=retry_if_result(is_bad))
# def get_number():
#     counter["counter"] += 1
#     print(f"Call: {counter['counter']}")

#     if counter["counter"] < 3:
#         return -1
#     return 10

# print(get_number())

from tenacity import retry, stop_after_attempt, before_sleep_log
import logging
from rich import print 
from rich.logging import RichHandler
from rich.table import Table
from rich.console import Console
from dotenv import load_dotenv
import time

load_dotenv()

logger = logging.getLogger("__file__")
logger.setLevel(logging.INFO)

console_handler = RichHandler()
console_handler.setLevel(logging.INFO)

file_handler = logging.FileHandler("retry_log.log", mode="a", encoding="utf-8")
file_handler.setLevel(logging.INFO)

formatter = logging.Formatter(
    "%(asctime)s - %(name)s - %(levelname)s - %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S"
)

file_handler.setFormatter(formatter)

logger.addHandler(console_handler)
logger.addHandler(file_handler)

console = Console()

@retry(stop=stop_after_attempt(3), before_sleep=before_sleep_log(logger, logging.WARNING))
def fail_with_log():
    print("Trying...")
    raise Exception("Oops")

fail_with_log()


Trying...

[04/01/26 13:27:15] WARNING  Retrying __main__.fail_with_log in 0 seconds as it raised           ]8;id=703692;file://c:\Users\10706030\Desktop\iamtest2\.venv\Lib\site-packages\tenacity\before_sleep.py\before_sleep.py]8;;\:]8;id=805097;file://c:\Users\10706030\Desktop\iamtest2\.venv\Lib\site-packages\tenacity\before_sleep.py#64\64]8;;\
                             Exception: Oops.                                                                      

Trying...

                    WARNING  Retrying __main__.fail_with_log in 0 seconds as it raised           ]8;id=412685;file://c:\Users\10706030\Desktop\iamtest2\.venv\Lib\site-packages\tenacity\before_sleep.py\before_sleep.py]8;;\:]8;id=282762;file://c:\Users\10706030\Desktop\iamtest2\.venv\Lib\site-packages\tenacity\before_sleep.py#64\64]8;;\
                             Exception: Oops.                                                                      

Trying...

RetryError: RetryError[<Future at 0x1faacc352d0 state=finished raised Exception>]